# 01 — Data Pipeline

Loads raw e-commerce review CSV, cleans and preprocesses the review text, and saves the result as a compressed `.npz` file for downstream modeling.

In [ ]:
import re
import os
import numpy as np
import pandas as pd
import nltk
from pathlib import Path

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Paths — project root is one level above notebooks/
PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

print('Project root:', PROJECT_ROOT)
print('Raw data dir:', RAW_DIR)
print('Processed dir:', PROCESSED_DIR)

## 1. Load Raw Data

Place the downloaded Kaggle CSV inside `data/raw/` before running this cell.

In [ ]:
CSV_FILE = RAW_DIR / 'customer_reviews_data.csv'

assert CSV_FILE.exists(), f'File not found: {CSV_FILE}'

df = pd.read_csv(CSV_FILE)
print(f'Loaded: {CSV_FILE.name}  |  Shape: {df.shape}')

## 2. Explore the Data

In [ ]:
print('Columns:', df.columns.tolist())
print()
print('Data types:')
print(df.dtypes)
print()
print('Null counts:')
print(df.isnull().sum())

In [ ]:
df.head(3)

## 3. Configure Column Names

The dataset uses `ReviewText` as the review body and `ReviewID` as the row identifier. Update the variables below if running on a different CSV.

In [ ]:
REVIEW_TEXT_COL = 'ReviewText'   # column containing the review body
REVIEW_ID_COL   = 'ReviewID'     # unique row identifier
PRODUCT_ID_COL  = 'ProductID'    # product identifier (used for ground-truth join in notebook 02)
RATING_COL      = None           # no rating column in this dataset

assert REVIEW_TEXT_COL in df.columns, (
    f"Column '{REVIEW_TEXT_COL}' not found. Available: {df.columns.tolist()}"
)

## 4. Filter — Remove Null / Empty Reviews

In [ ]:
before = len(df)
df = df[df[REVIEW_TEXT_COL].notna()]
df = df[df[REVIEW_TEXT_COL].astype(str).str.strip().ne('')]
df = df.reset_index(drop=True)
print(f'Rows before: {before}  |  After removing nulls/empty: {len(df)}')

## 5. Text Cleaning

Lowercase → strip HTML tags → remove URLs → strip non-alphabetic characters → collapse whitespace.

In [ ]:
_html_tag = re.compile(r'<[^>]+>')
_url      = re.compile(r'https?://\S+|www\.\S+')
_non_alpha = re.compile(r'[^a-z\s]')
_whitespace = re.compile(r'\s+')

def clean_text(text: str) -> str:
    text = str(text).lower()
    text = _html_tag.sub(' ', text)
    text = _url.sub(' ', text)
    text = _non_alpha.sub(' ', text)
    text = _whitespace.sub(' ', text).strip()
    return text

df['cleaned'] = df[REVIEW_TEXT_COL].apply(clean_text)
print('Sample cleaned reviews:')
df[['cleaned']].head(3)

## 6. Tokenization & Stopword Removal

In [ ]:
STOP_WORDS = set(stopwords.words('english'))

def tokenize_and_filter(text: str) -> list[str]:
    tokens = word_tokenize(text)
    return [t for t in tokens if t not in STOP_WORDS and len(t) > 2]

df['tokens'] = df['cleaned'].apply(tokenize_and_filter)
print('Sample tokens:')
df['tokens'].head(3)

## 7. Lemmatization

In [ ]:
lemmatizer = WordNetLemmatizer()

def lemmatize(tokens: list[str]) -> str:
    return ' '.join(lemmatizer.lemmatize(t) for t in tokens)

df['processed'] = df['tokens'].apply(lemmatize)

# Drop rows that became empty after processing
df = df[df['processed'].str.strip().ne('')].reset_index(drop=True)
print(f'Final row count: {len(df)}')
print('Sample processed reviews:')
df[['processed']].head(3)

## 8. Save as Compressed `.npz`

In [ ]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
out_path = PROCESSED_DIR / 'reviews.npz'

save_kwargs = {'reviews': df['processed'].to_numpy(dtype=str)}

if REVIEW_ID_COL and REVIEW_ID_COL in df.columns:
    save_kwargs['ids'] = df[REVIEW_ID_COL].to_numpy()

if PRODUCT_ID_COL and PRODUCT_ID_COL in df.columns:
    save_kwargs['product_ids'] = df[PRODUCT_ID_COL].to_numpy()

if RATING_COL and RATING_COL in df.columns:
    save_kwargs['ratings'] = df[RATING_COL].to_numpy()

np.savez_compressed(out_path, **save_kwargs)
print(f'Saved {len(df)} reviews to {out_path}')
print('Arrays stored:', list(save_kwargs.keys()))